# 🚀 S4D-RUPA-CLIP: Weakly-Supervised Video Anomaly Detection
### Standalone Residual-to-Text Counterfactual Alignment with S4D Temporal Modeling
**Tác giả:** Huỳnh Quang Vũ (MSSV: 2351010247)  
**Giảng viên hướng dẫn:** TS. Lê Viết Tuấn  
**Trường Đại học Mở TP. Hồ Chí Minh (HCMCOU)**  

---
### 🎯 Mục tiêu khoa học & Hiệu năng:
- **UCF-Crime:** Frame-level ROC-AUC >= 90.0%
- **XD-Violence:** Average Precision (AP) >= 87.0%
- **100% Standalone (Train from Scratch):** Triệt tiêu hoàn toàn DSANet, Dual GCN, Local Transformer và Warm-Start.
- **Kế thừa chuẩn hóa dữ liệu từ RUPA-DSA_final:** Đồng bộ và lập bản đồ đường dẫn tuyệt đối chính xác 100% trước khi huấn luyện.

In [ ]:
# ========================= 1. CẤU HÌNH & TẢI MÃ NGUỒN =========================
DATASET = "ucf"             # Chọn "ucf" hoặc "xd"
MAX_EPOCHS = 15            # Số epoch huấn luyện
BATCH_SIZE = 64            # Kích thước batch
NUM_WORKERS = 0            # Single-process data feeding (ổn định tuyệt đối)
SEED = 234
REPO_URL = "https://github.com/Sharvuz/S4D-RUPA-CLIP.git"

import json, os, random, shutil, subprocess, sys, time, zipfile
from pathlib import Path

WORK = Path("/kaggle/working")
INPUT = Path("/kaggle/input")
REPO = WORK / "S4D-RUPA-CLIP"
ARTIFACTS = WORK / "s4d_rupa_artifacts"
ARTIFACTS.mkdir(parents=True, exist_ok=True)

# Clone hoặc Pull code mới nhất
if not REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "pull"], check=False)

os.chdir(str(REPO))

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "ftfy", "regex", "scikit-learn", "pandas", "git+https://github.com/openai/CLIP.git"
], check=True)

import numpy as np
import pandas as pd
import torch

print("Repository:", REPO_URL)
print("PyTorch:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Model:", torch.cuda.get_device_name(0))
    print("GPU Count:", torch.cuda.device_count())
assert torch.cuda.is_available(), "Bật Accelerator = GPU trong Kaggle Settings!"


In [ ]:
# ========================= 2. CHUẨN HÓA DỮ LIỆU & LẬP BẢN ĐỒ (CHUẨN RUPA) =========================
print("🔍 Đang quét toàn bộ hệ thống Kaggle để tìm file .npy...")
t0 = time.time()

# Quét sạch sẽ toàn bộ /kaggle/input và /kaggle/working
all_npy_paths = list(INPUT.rglob("*.npy")) + list(WORK.rglob("*.npy"))

if not all_npy_paths:
    raise FileNotFoundError("❌ CẢNH BÁO ĐỎ: Không tìm thấy file .npy nào! Hãy Add Input UCFClipFeatures vào Kaggle.")
else:
    print(f"✅ Radar đã quét thấy {len(all_npy_paths)} file .npy trên hệ thống trong {time.time()-t0:.2f}s!")

# Tạo từ điển bản đồ: Tên file -> Đường dẫn tuyệt đối trên Kaggle
file_map = {path.name: path for path in all_npy_paths}

def rewrite_csv(csv_rel, output_name):
    input_csv = REPO / csv_rel
    output = ARTIFACTS / output_name
    print(f"Đang đồng bộ danh sách {input_csv.name}...")
    
    missing = []
    is_first_line = True
    
    with input_csv.open("r", encoding="utf-8") as f, output.open("w", encoding="utf-8") as out:
        for line in f:
            line = line.strip()
            if not line: continue
            
            parts = line.split(",")
            feature_path = parts[0]
            
            if is_first_line and ("path" in feature_path.lower() or "name" in feature_path.lower()):
                out.write(line + "\n")
                is_first_line = False
                continue
            is_first_line = False
            
            feature_basename = Path(feature_path).name
            
            # Tự động khớp các biến thể đặt tên ký tự (# sang +, _, rỗng)
            possible_names = [
                feature_basename,
                feature_basename.replace("#", "+"),
                feature_basename.replace("+", "#"),
                feature_basename.replace("#", ""),
                feature_basename.replace("#", "_"),
            ]
            
            match_found = False
            for p_name in possible_names:
                if p_name in file_map:
                    parts[0] = str(file_map[p_name])
                    out.write(",".join(parts) + "\n")
                    match_found = True
                    break
                    
            if not match_found:
                missing.append(feature_basename)
                
    if missing:
        raise FileNotFoundError(f"❌ Lỗi: Thiếu {len(missing)} file video trên Kaggle. Ví dụ: {missing[:5]}")
        
    print(f"  -> {output_name}: Đồng bộ thành công 100%! Không thiếu file nào.")
    return output

# Đồng bộ train và test CSV
train_csv = rewrite_csv(f"list/{DATASET}_CLIP_rgb.csv", f"{DATASET}_train.csv")
test_csv = rewrite_csv(f"list/{DATASET}_CLIP_rgbtest.csv", f"{DATASET}_test.csv")

# Preflight kiểm tra đọc thử 1 file mẫu từ đĩa
sample_path = Path(pd.read_csv(train_csv).iloc[0]["path"])
sample = np.load(sample_path)
assert sample.ndim == 2 and sample.shape[-1] == 512, f"Lỗi shape: {sample.shape}"
print(f"🎉 Feature preflight hoàn hảo: {sample_path.name} | Shape: {sample.shape}")


In [ ]:
# ========================= 3. HUẤN LUYỆN S4D-RUPA-CLIP =========================
train_py = f"src/{DATASET}_train.py"
model_save_dir = str(ARTIFACTS / "models")
os.makedirs(model_save_dir, exist_ok=True)

cmd = [
    sys.executable, "-u", train_py,
    "--train-list", str(train_csv),
    "--test-list", str(test_csv),
    "--dataset", DATASET,
    "--batch-size", str(BATCH_SIZE),
    "--lr", "5e-5",
    "--max-epoch", str(MAX_EPOCHS),
    "--warmup-epochs", "2",
    "--num-layers", "2",
    "--d-state", "64",
    "--ssm-type", "s4d",
    "--num-prototypes", "16",
    "--num-workers", str(NUM_WORKERS),
    "--use-text-prompt", "True",
    "--loss-rec-weight", "0.5",
    "--loss-sem-weight", "0.5",
    "--loss-bg-weight", "0.1",
    "--save-dir", model_save_dir
]

print(f"🚀 BẮT ĐẦU HUẤN LUYỆN S4D-RUPA-CLIP TRÊN {DATASET.upper()} ({MAX_EPOCHS} Epochs)...\n")
subprocess.run(cmd, check=True)


In [ ]:
# ========================= 4. ĐÁNH GIÁ CHECKPOINT TỐT NHẤT =========================
test_py = f"src/{DATASET}_test.py"
best_model_path = os.path.join(model_save_dir, f"s4d_rupa_clip_{DATASET}_best.pth")

test_cmd = [
    sys.executable, "-u", test_py,
    "--test-list", str(test_csv),
    "--model-path", best_model_path,
    "--num-layers", "2",
    "--d-state", "64",
    "--ssm-type", "s4d",
    "--num-prototypes", "16",
    "--use-text-prompt", "True"
]

print("🔍 BẮT ĐẦU ĐÁNH GIÁ CHI TIẾT UCF-CRIME TEST SET...")
subprocess.run(test_cmd, check=True)
